# 05 — Cluster-Level Crops

## Goal

Extend the greedy crop defined on a reference structure (notebook 04) to every
other member of its cluster. For a cluster of K members and N reference crops,
this produces K × N total crops.

## Non-solvent correspondence

Structures in the same 100%-identity cluster have the same sequence. Non-solvent
residues are matched by `(entity_id, sym_id, local position within chain)`. This
gives an explicit correspondence between each reference residue and its counterpart
in each sample — the correspondence needed for Kabsch alignment.

Residues absent in a sample (missing density, truncated chain) are recorded as
`None` in `sample_ns_indices`. Kabsch alignment will skip those pairs.

## Water inclusion

Waters have no cross-structure correspondence. For each sample crop, waters are
included independently: all present water oxygens within `WATER_RADIUS` Å of the
sample's corresponding seed center atom (CA of the seed residue; falls back to
mean of matched non-solvent centers if the seed is absent).

**TODO to explore:** alternative water inclusion — within a smaller radius (e.g.
4–5 Å) of *any* polymer/ligand atom in the crop. This gives denser local
hydration shells and may be more relevant for conservation scoring.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../scripts')
sys.path.insert(0, '/data/rbg/users/gloriama/dev/foldeverything/src')

from pathlib import Path
import numpy as np

from cluster_filter import load_manifest_index, filter_clusters
from crop import (
    count_nonsolvent_residues,
    greedy_crop,
    crop_sample,
    save_crop_as_mmcif,
)
from boltzgen.data.data import Structure

CLUSTER_FILE   = '../clusters-by-entity-100.txt'
MANIFEST_FILE  = '/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/manifest.json'
STRUCTURES_DIR = Path('/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures')
OUTPUT_DIR     = Path('/data/rbg/users/gloriama/dev/water_conservation/outputs/crops')

RADIUS       = 20.0  # Å — greedy crop sphere radius
WATER_RADIUS = 20.0  # Å — water inclusion radius around seed center

## Section 1 — Load cluster and define reference crops

Reuse the same cluster as notebook 04 (first 5-member cluster at 2.0 Å).
The reference is the member with the most non-solvent residues.

In [ ]:
manifest_index = load_manifest_index(MANIFEST_FILE)
filtered, _ = filter_clusters(
    manifest_index, resolution_cutoff=2.0, min_cluster_size=5, cluster_file=CLUSTER_FILE
)

cluster = [c for c in filtered if 5 <= len(c) <= 8][0]
print(f'Cluster ({len(cluster)} members): {cluster}')

counts = {
    pdb: count_nonsolvent_residues(np.load(STRUCTURES_DIR / f'{pdb}.npz', allow_pickle=True))
    for pdb in cluster
}
ref_pdb = max(counts, key=counts.__getitem__)
print(f'Reference: {ref_pdb}  ({counts[ref_pdb]} non-solvent residues)')

In [ ]:
ref_struct = Structure.load(STRUCTURES_DIR / f'{ref_pdb}.npz')
ref_crops  = greedy_crop(ref_struct, radius=RADIUS)
print(f'Reference crops: {len(ref_crops)}')
print(f'  (radius={RADIUS} A, seeds = non-water residues only)')

## Section 2 — Crop all cluster members

For each sample, `crop_sample` matches non-solvent residues by
`(entity_id, sym_id, local position)` and collects nearby waters independently.

In [ ]:
structs = {pdb: Structure.load(STRUCTURES_DIR / f'{pdb}.npz') for pdb in cluster}

# all_crops[pdb][crop_idx] = crop_sample result dict
all_crops = {}
for pdb in cluster:
    sample_crops = [
        crop_sample(ref_struct, rc, structs[pdb], water_radius=WATER_RADIUS)
        for rc in ref_crops
    ]
    all_crops[pdb] = sample_crops
    n_fallback = sum(c['used_seed_fallback'] for c in sample_crops)
    print(f'{pdb}: {len(sample_crops)} crops  '
          f'(seed fallback used in {n_fallback}/{len(sample_crops)} crops)')

## Section 3 — Statistics

For each reference crop: how many residues are matched vs missing per sample,
and how many waters are captured.

In [ ]:
print(f'{'Crop':>4}  Ref_NS  ' + '  '.join(f'{p:>8}' for p in cluster))
print(f'          (ns)   ' + '  '.join(f'{"match/wat":>8}' for _ in cluster))
print('-' * (12 + len(cluster) * 10))

for ci, rc in enumerate(ref_crops):
    ref_ns = rc['n_nonsolvent']
    cells = []
    for pdb in cluster:
        sc = all_crops[pdb][ci]
        cells.append(f'{sc["n_matched"]}/{sc["n_water"]:>3}')
    print(f'{ci:>4}  {ref_ns:>6}  ' + '  '.join(f'{c:>8}' for c in cells))

print()
print('Columns show matched_nonsolvent / waters for each sample.')
print(f'Total crops: {len(ref_crops)} reference x {len(cluster)} samples = {len(ref_crops)*len(cluster)}')

## Section 4 — Save all crops as mmCIF

Layout: `outputs/crops/<pdb>/radius_<R>_crop_<nn>.cif`

Each file contains the matched non-solvent residues plus nearby waters for that
sample. Empty crops (0 matched non-solvent residues) are skipped with a note.

In [ ]:
saved = 0
skipped = 0

for pdb in cluster:
    for ci, sc in enumerate(all_crops[pdb]):
        matched_ns = [r for r in sc['sample_ns_indices'] if r is not None]
        all_indices = matched_ns + sc['sample_water_indices']

        if not all_indices:
            print(f'  [skip] {pdb} crop {ci:02d} — no residues')
            skipped += 1
            continue

        out = OUTPUT_DIR / pdb / f'radius_{WATER_RADIUS}_crop_{ci:02d}.cif'
        save_crop_as_mmcif(structs[pdb], all_indices, out)
        saved += 1

print(f'Saved {saved} crops, skipped {skipped} empty crops.')
print(f'Output directory: {OUTPUT_DIR}')